# step 2 — granite 방향 뒤집기 (옵션 B: target=snake)

**대응 RQ:** RQ2 — granite가 **자기 선호 방향(snake)**에선 step1 기제(국소·Value)를 보이는가.

앞선 진단: granite는 camelCase를 안 따름(snake 프라이어 지배) → target=camel에선 천장이 없어 기제 관측 불가(negative, `docs/step2/granite/results.md`). 여기선 **지침=snake / 선행=camel(위반)**으로 뒤집어, "위반(camel)→준수(snake) 회복"의 국소·Value 기제가 나타나는지 본다.

- 조건: pos-**snake**-weak, POOL n=0(선행 전부 위반=**camel**), token_unit='last', seed 0–9.
- 결과: `results/step2_granite_snake/`. (camel-target 결과와 분리)
- v 코사인은 target과 무관(같은 이름 camel/snake 방향차)하므로 **재실행 생략** — camel-target 코사인과 동일.

> **sanity 관건:** 이번엔 S_깨끗(전부 snake 선행+snake 지침)이 **강하게 양수**여야 함(granite가 snake는 잘 함). 그리고 S_위반(camel 선행)이 그보다 낮아야 회복을 잴 수 있음. 둘 다 아니면 granite는 방향 무관하게 프라이어 고정.


In [ ]:
!pip install -q transformers accelerate torch matplotlib pandas
import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
SEED=0; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin step2/granite-3b-code
!git checkout step2/granite-3b-code
!git pull --quiet origin step2/granite-3b-code
!pip install -e . -q
import sys; sys.path.insert(0,'src')
!git log --oneline -1

In [ ]:
# 조건 — target=SNAKE로 뒤집기. 선행 전부 위반(=camel). token_unit='last'.
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation,
                                Intervention, InterventionKind)
MODEL = ModelSpec(name='ibm-granite/granite-3b-code-instruct-2k', family='granite', dtype='float16')
DONORS=['compliant','unrelated_snake']   # 준수=snake이므로 무관통제는 unrelated_snake(형태통제)
SEEDS=list(range(10)); STEP='step2_granite_snake'; TOKEN_UNIT='last'

def _pre(): return PrecedingCode(n_compliant=0, n_functions=12, composition=Composition.POOL)
def _ins(): return Instruction(form=InstructionForm.POSITIVE, target_notation=Notation.SNAKE)  # ← 뒤집음
def sweep_cond(donor,s):
    return Condition(model=MODEL, preceding=_pre(), instruction=_ins(),
                     intervention=Intervention(kind=InterventionKind.KEY_VALUE, layers='sweep', donor=donor),
                     seed=s, token_unit=TOKEN_UNIT)
sweep_conditions=[sweep_cond(d,s) for d in DONORS for s in SEEDS]
PREDICTION=('방향 뒤집기(target=snake). granite가 선호 방향(snake)에선 위반(camel)→준수(snake) '
            '회복의 국소·Value 기제를 보이는가. 보이면 "기제 동일, 방향은 프라이어 따름".')
print(len(sweep_conditions),'스윕 (target=snake) | STEP=',STEP)

In [ ]:
# 셀 4 — SANITY. n_sub>0 AND S_깨끗(snake)>S_위반 이어야 PASS.
from harness import run, ResultRecord, save_result, result_path
from harness.model import load_model
handle=load_model(MODEL); gqa=handle.gqa_info()
print(f'layers={handle.num_layers} kv={gqa.num_key_value_heads} group={gqa.group_size}')
c0=sweep_cond('compliant',0); out0=run(c0, handle=handle)
save_result(ResultRecord(condition=out0.condition, metrics=out0.metrics, step=STEP, rq='RQ2', prediction=PREDICTION))
ex=out0.metrics.extra
print(f'정렬토큰 {ex["n_substituted_tokens"]} 스킵 {len(ex["skipped_names"])}')
print(f'S_깨끗(snake) {ex["S_clean"]:+.3f} | S_위반(camel선행) {ex["S_base"]:+.3f}')
aligned=ex['n_substituted_tokens']>0
sflip=(ex['S_clean'] is not None and ex['S_clean']==ex['S_clean'] and ex['S_clean']>ex['S_base'])
print('\n>>> SANITY', 'PASS -> 셀 5' if (aligned and sflip) else 'FAIL')
print('   해석: S_깨끗 강한 양수 & S_위반 더 낮음 => granite도 snake 방향엔 절벽 존재(기제 잴 수 있음)')
print('        S_위반도 여전히 높음 => camel 선행이 granite를 못 흔듦(방향 무관 프라이어 고정)')

In [ ]:
# 셀 5 — 전체 실행. 재개 가능.
new=skipped=0
for i,c in enumerate(sweep_conditions,1):
    if result_path(c, step=STEP).exists(): skipped+=1
    else:
        out=run(c, handle=handle)
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics, step=STEP, rq='RQ2', prediction=PREDICTION)); new+=1
    if i%4==0 or i==len(sweep_conditions): print(f'[{i}/{len(sweep_conditions)}] 새 {new} 건너뜀 {skipped}')
print('완료.')

In [ ]:
# 셀 6 — 요약: 방향 뒤집기 결과 (피크·K/V). camel-target(negative)과 대조.
import numpy as np, pandas as pd
from collections import defaultdict
from harness import result_path
from harness.results import load_result
from harness.intervention import peak_layer
recs=[load_result(result_path(c, step=STEP)) for c in sweep_conditions if result_path(c, step=STEP).exists()]
NL=handle.num_layers; KINDS=['key','value','key_value']
dpres=sorted({r.condition.intervention.donor for r in recs})
sc=np.mean([r.metrics.extra['S_clean'] for r in recs]); sb=np.mean([r.metrics.extra['S_base'] for r in recs])
print(f'로드 {len(recs)}  NL={NL}  정렬토큰={recs[0].metrics.extra["n_substituted_tokens"]}')
print(f'S_깨끗(snake) {sc:+.2f} > S_위반(camel선행) {sb:+.2f} : {sc>sb}   (범위 {sc-sb:+.2f})')
agg={d:{k:defaultdict(list) for k in KINDS} for d in dpres}
for r in recs:
    d=r.condition.intervention.donor
    for L,flat in r.metrics.per_layer.items():
        for k in KINDS:
            if f'{k}__recovery' in flat: agg[d][k][int(L)].append(flat[f'{k}__recovery'])
print('=== 피크 (donor x kind), rel=peak/(NL-1) ===')
rows=[]
for d in dpres:
    for k in KINDS:
        cur={L:float(np.mean(v)) for L,v in agg[d][k].items()}
        pk=peak_layer(cur); rows.append({'donor':d,'kind':k,'peak_L':pk[0],'rel':round(pk[0]/(NL-1),3),'peak_rec':round(pk[1],3)})
print(pd.DataFrame(rows).to_string(index=False))
print('\n해석: peak_rec가 유의(>0.2)하고 Value 우세면 => granite도 기제 동일, 방향만 프라이어.')
print('      camel-target(0.12, Value우세X)과 대비해 서술.')

In [ ]:
import shutil
shutil.make_archive('step2_granite_snake_results','zip','results/step2_granite_snake')
try:
    from google.colab import files; files.download('step2_granite_snake_results.zip')
except Exception as e:
    print('Colab 아님:', e)